# Qwen 3.5 4B через llama.cpp


> Ноутбук предполагает, что он лежит в корне проекта рядом с папками `llama.cpp` и `models`.

## Запуск модели

In [ ]:
import subprocess
import time
from urllib.request import urlopen
from urllib.error import URLError

HOST = "127.0.0.1"
PORT = 8080
HEALTH_URL = f"http://{HOST}:{PORT}/health"


def is_server_ready() -> bool:
    try:
        with urlopen(HEALTH_URL, timeout=1):
            return True
    except URLError:
        return False


def stop_existing_llama_server() -> None:
    command = f"""
    $connection = Get-NetTCPConnection `
        -LocalAddress "{HOST}" `
        -LocalPort {PORT} `
        -State Listen `
        -ErrorAction SilentlyContinue |
        Select-Object -First 1

    if (-not $connection) {{
        Write-Output "NOT_FOUND"
        exit
    }}

    $process = Get-Process `
        -Id $connection.OwningProcess `
        -ErrorAction SilentlyContinue

    if (-not $process) {{
        Write-Output "NOT_FOUND"
        exit
    }}

    if ($process.ProcessName -ne "llama-server") {{
        Write-Output "WRONG_PROCESS:$($process.ProcessName):$($process.Id)"
        exit
    }}

    Stop-Process -Id $process.Id -Force
    Write-Output "STOPPED:$($process.Id)"
    """

    result = subprocess.run(
        ["powershell", "-NoProfile", "-Command", command],
        capture_output=True,
        text=True,
        check=False,
    )

    output = result.stdout.strip()

    if output.startswith("STOPPED:"):
        pid = output.split(":", 1)[1]
        print(f"Предыдущий llama-server остановлен, PID: {pid}")

    elif output.startswith("WRONG_PROCESS:"):
        _, process_name, pid = output.split(":", 2)
        raise RuntimeError(
            f"Порт {PORT} занят другим процессом: "
            f"{process_name}, PID: {pid}"
        )

    elif output == "NOT_FOUND":
        print("Запущенный llama-server не найден")

    else:
        raise RuntimeError(
            "Не удалось проверить процесс на порту "
            f"{PORT}.\n{result.stderr.strip()}"
        )


def wait_until_stopped(timeout: float = 10) -> None:
    deadline = time.time() + timeout

    while time.time() < deadline:
        if not is_server_ready():
            return
        time.sleep(0.2)

    raise RuntimeError(
        f"llama-server не остановился за {timeout} секунд"
    )


def start_llama_server() -> subprocess.Popen:
    server = subprocess.Popen([
        r".\llama.cpp\llama-server.exe",
        "--model", r".\models\Qwen_Qwen3.5-4B-Q4_K_M.gguf",
        "--alias", "qwen3.5-4b",
        "--host", HOST,
        "--port", str(PORT),
        "--ctx-size", "4096",
        "--threads", "4",
        "--threads-batch", "4",
        "--reasoning", "off",
    ])

    deadline = time.time() + 120

    while time.time() < deadline:
        if server.poll() is not None:
            raise RuntimeError(
                f"llama-server завершился с кодом "
                f"{server.returncode}"
            )

        if is_server_ready():
            print(
                f"llama-server запущен и готов, PID: {server.pid}"
            )
            return server

        time.sleep(1)

    server.terminate()
    raise TimeoutError(
        "llama-server не запустился за 120 секунд"
    )


if is_server_ready():
    print("llama-server уже запущен — перезапускаю")
    stop_existing_llama_server()
    wait_until_stopped()
else:
    print("Работающий llama-server не обнаружен")

server = start_llama_server()

Работающий llama-server не обнаружен
llama-server запущен и готов, PID: 5788


## Диалог с моделью

In [10]:
from openai import OpenAI


client = OpenAI(
    base_url="http://127.0.0.1:8080/v1",
    api_key="not-needed",
    timeout=300.0,
)

print("Отправляю запрос модели...")

response = client.chat.completions.create(
    model="qwen3.5-4b",
    messages=[
        {
            "role": "system",
            "content": (
                "Ты преобразуешь неструктурированные данные "
                "в строго структурированный результат."
            ),
        },
        {
            "role": "user",
            "content": (
                "Преобразуй запись в JSON: "
                "дата 15.07.2026, артикул A-17, "
                "сумма 1 250,50 руб."
            ),
        },
    ],
    temperature=0.1,
    max_tokens=256,
)

text = response.choices[0].message.content

if text is None:
    raise RuntimeError("Модель вернула пустой ответ")

print("\nОтвет модели:\n")
print(text)

Отправляю запрос модели...

Ответ модели:

```json
{
  "date": "2026-07-15",
  "article": "A-17",
  "amount": 1250.50,
  "currency": "RUB"
}
```


In [11]:
text = response.choices[0].message.content

if text is None:
    raise RuntimeError("Модель вернула пустой ответ")

print(text)

timings = response.model_extra.get("timings", {})
usage = response.usage

print("\nСтатистика:")

print(f"Токены промпта:      {usage.prompt_tokens}")
print(f"Токены генерации:    {usage.completion_tokens}")
print(f"Всего токенов:       {usage.total_tokens}")

print(f"Время префилла:      {timings['prompt_ms'] / 1000:.2f} с")
print(f"Скорость префилла:   {timings['prompt_per_second']:.2f} ток/с")

print(f"Время генерации:     {timings['predicted_ms'] / 1000:.2f} с")
print(f"Скорость генерации:  {timings['predicted_per_second']:.2f} ток/с")

```json
{
  "date": "2026-07-15",
  "article": "A-17",
  "amount": 1250.50,
  "currency": "RUB"
}
```

Статистика:
Токены промпта:      74
Токены генерации:    60
Всего токенов:       134
Время префилла:      0.92 с
Скорость префилла:   80.61 ток/с
Время генерации:     4.99 с
Скорость генерации:  12.03 ток/с


## Остановка модели

Выполни эту ячейку, когда модель больше не нужна.

In [12]:
server = globals().get("server")

if server is None:
    print("Нет ссылки на процесс сервера — останавливать нечего.")
elif server.poll() is not None:
    print(f"Сервер уже остановлен. Код завершения: {server.returncode}")
else:
    server.terminate()

    try:
        server.wait(timeout=10)
        print("Сервер успешно остановлен.")
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait()
        print("Сервер не завершился вовремя и был принудительно остановлен.")

Сервер успешно остановлен.
